# CyberTwin Milestone 4.1: Corrected Digital Twin & Temporal Modeling

This notebook demonstrates the corrected temporal simulation methodology for CyberTwin.

## 1. Problem with Initial Temporal Simulation
In the original Milestone 4, temporal sequences were generated by slicing randomized datasets into windows of 20 and labeling the sequence as ATTACK if *any* flow in the window was an attack. Because ~42% of the data was PortScan, $1 - (1 - 0.42)^20 \approx 99.99\%$. Thus, almost every sequence was labeled ATTACK, leading to 0 Specificity and an invalid evaluation.

## 2. Corrected Simulation Methodology
Instead of random windows, we construct explicit, balanced *scenarios* by tracking original row indices to ensure zero contamination between Train/Validation/Test sets.
Scenarios:
- **NORMAL** (0): 100% Benign
- **ATTACK ONSET** (1): 50% Benign -> 50% Attack
- **SUSTAINED ATTACK** (1): 100% Attack
- **ATTACK RECOVERY** (0): 50% Attack -> 50% Benign
This provides a rigorous sequence-level task (predicting the scenario state) rather than exploiting a trivial probability.

In [ ]:
import sys
sys.path.append('..')
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display
import warnings
warnings.filterwarnings('ignore')

## 3. Sequence Statistics

In [ ]:
manifest = pd.read_csv('../results/milestone_4_1/tables/temporal_dataset_manifest.csv')
display(manifest)
audit = pd.read_csv('../results/milestone_4_1/tables/sequence_source_audit.csv')
print("Sequence distribution in TEST set:")
display(audit[audit['source_partition'] == 'TEST']['scenario'].value_counts())

## 4. Scenario Visualizations

In [ ]:
display(Image(filename='../results/milestone_4_1/figures/scenario_normal.png'))
display(Image(filename='../results/milestone_4_1/figures/scenario_attack_onset.png'))
display(Image(filename='../results/milestone_4_1/figures/scenario_sustained_attack.png'))
display(Image(filename='../results/milestone_4_1/figures/scenario_attack_recovery.png'))

## 5. Temporal Model Training (Temporal BiLSTM)

In [ ]:
history_df = pd.read_csv('../results/milestone_4_1/metrics/temporal_training_history.csv')
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history_df['epoch'], history_df['train_loss'], label='Train Loss')
plt.plot(history_df['epoch'], history_df['val_loss'], label='Val Loss')
plt.legend()
plt.title('Loss')
plt.subplot(1, 2, 2)
plt.plot(history_df['epoch'], history_df['train_f1'], label='Train F1')
plt.plot(history_df['epoch'], history_df['val_f1'], label='Val F1')
plt.legend()
plt.title('F1 Score')
plt.show()

## 6. Test Evaluation & Confusion Matrix

In [ ]:
metrics = pd.read_csv('../results/milestone_4_1/tables/corrected_model_comparison.csv')
display(metrics)
display(Image(filename='../results/milestone_4_1/figures/temporal_confusion_matrix.png'))

## 7. Scenario-Specific Results

In [ ]:
scen_perf = pd.read_csv('../results/milestone_4_1/tables/scenario_performance.csv')
display(scen_perf)

## 8. Digital Twin State Transitions & Risk Timeline

In [ ]:
log_df = pd.read_csv('../results/milestone_4_1/tables/digital_twin_state_log.csv')
plt.figure(figsize=(14, 4))
plt.plot(log_df['simulation_step'], log_df['risk_score'], color='red')
plt.title('Digital Twin Network Risk Score (Exponential Moving Average)')
plt.xlabel('Simulation Step')
plt.ylabel('Risk Score')
plt.fill_between(log_df['simulation_step'], log_df['risk_score'], color='red', alpha=0.2)
plt.show()
display(log_df.head())

## 9. SHAP Random Forest Explanation
Due to architecture limitations of the PyTorch `GradientExplainer` with BiLSTMs, the primary explainability is provided by a SHAP `TreeExplainer` on the original Random Forest flow-level model.

In [ ]:
display(Image(filename='../results/milestone_4_1/figures/shap_rf_bar.png'))
display(Image(filename='../results/milestone_4_1/figures/shap_rf_summary.png'))

## 10. Old vs Corrected Temporal Comparison

In [ ]:
comp = pd.read_csv('../results/milestone_4_1/tables/old_vs_corrected_temporal.csv')
display(comp)